# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adeenafatima0/ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

Before writing my rule, I checked two signals it depends on:

**Signal 1 — staleness (behind FlyRank's refresh flags):** I expected stale pages (180+ days since update) to decline more than fresh pages. Verdict: OPPOSITE. Non-stale pages actually declined more (54.2%) than stale pages (47.1%). Also worth flagging: only 174 out of 30,000 pages even qualify as "stale" at this threshold, so this bucket is tiny and the signal is weak either way. This is a useful negative — it tells me staleness alone isn't a reliable driver of decline in this dataset, so I shouldn't lean on it heavily in my rule.

**Signal 2 — visibility (behind quick-win logic):** I expected visible pages (500+ impressions) to show a clearer decline signal than low-visibility pages. Verdict: CONFIRMED. Visible pages declined at 59.6% vs 47.5% for low-visibility pages — a meaningful gap, and it makes sense since visible pages actually have enough traffic data for a trend to be measurable at all.

Given this, my rule leans mainly on **visibility + current decline**, and treats staleness as a weaker, secondary signal rather than a primary driver — since the data doesn't support staleness alone predicting decline well.

Rule (plain words): flag a page as a review candidate if it's visible (500+ impressions) AND currently declining. Reason codes:
- `visible_declining`: visible + trend_direction == "down" → priority review
- `visible_stable`: visible but not currently declining → lower priority, worth monitoring
- `stale_but_weak_signal`: stale, regardless of trend → flagged separately, low confidence, since staleness alone didn't prove predictive

In [4]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/adeenafatima0/ml-internship"
REPO_DIR = "ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

print("Working dir:", os.getcwd())
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Signal 1: staleness vs decline rate
df["is_stale"] = df["days_since_last_update"] >= 180
stale_bucket = df.groupby("is_stale")["trend_direction"].apply(lambda x: (x.str.lower() == "down").mean())
stale_n = df.groupby("is_stale").size()
print("Decline rate by staleness bucket:")
print(stale_bucket)
print("\nRow counts:")
print(stale_n)

# Signal 2: visibility vs decline rate
df["is_visible"] = df["impressions_90d"] >= 500
visible_bucket = df.groupby("is_visible")["trend_direction"].apply(lambda x: (x.str.lower() == "down").mean())
visible_n = df.groupby("is_visible").size()
print("\nDecline rate by visibility bucket:")
print(visible_bucket)
print("\nRow counts:")
print(visible_n)

Working dir: /content/ml-internship/ml-internship
Decline rate by staleness bucket:
is_stale
False    0.542480
True     0.471264
Name: trend_direction, dtype: float64

Row counts:
is_stale
False    29826
True       174
dtype: int64

Decline rate by visibility bucket:
is_visible
False    0.474687
True     0.595540
Name: trend_direction, dtype: float64

Row counts:
is_visible
False    13274
True     16726
dtype: int64


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*## 2. Build the ranked queue (writes the CSV)

Scoring visible + declining pages higher, stale pages get a smaller separate bump since that signal was weak. Writing the ranked queue to work/outputs/baseline_action_score.csv.

In [5]:
import numpy as np

df["is_declining"] = df["trend_direction"].str.lower() == "down"

# Reason code logic
def get_reason_code(row):
    if row["is_visible"] and row["is_declining"]:
        return "visible_declining"
    elif row["is_visible"] and not row["is_declining"]:
        return "visible_stable"
    elif row["is_stale"]:
        return "stale_but_weak_signal"
    else:
        return "low_priority"

def get_action(reason):
    return {
        "visible_declining": "refresh_review",
        "visible_stable": "monitor",
        "stale_but_weak_signal": "low_confidence_review",
        "low_priority": "no_action"
    }[reason]

df["reason_code"] = df.apply(get_reason_code, axis=1)
df["action"] = df["reason_code"].apply(get_action)

# Score: visibility + declining matters most, staleness gets a small separate weight
df["baseline_score"] = (
    df["is_visible"].astype(int) * 2
    + df["is_declining"].astype(int) * 3
    + df["is_stale"].astype(int) * 1
)

# Build the ranked queue
queue = df[["content_id", "baseline_score", "reason_code", "action",
            "impressions_90d", "days_since_last_update", "trend_direction"]]
queue = queue.sort_values("baseline_score", ascending=False)

import os
os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

print("Rows written:", len(queue))
print(queue["reason_code"].value_counts())
queue.head(10)

Rows written: 30000
reason_code
low_priority             13117
visible_declining         9961
visible_stable            6765
stale_but_weak_signal      157
Name: count, dtype: int64


,content_id,baseline_score,reason_code,action,impressions_90d,days_since_last_update,trend_direction
698,content_b16bd7307b39,6,visible_declining,refresh_review,4590,194,down
7021,content_1bfaa38ff26c,6,visible_declining,refresh_review,25715,194,down
3507,content_074ba6ead17b,6,visible_declining,refresh_review,533,183,down
16751,content_cf56e2e2e282,6,visible_declining,refresh_review,61678,194,down
26799,content_77d4d5930e5e,6,visible_declining,refresh_review,828,194,down
16514,content_7368877ea310,6,visible_declining,refresh_review,59472,194,down
5327,content_fe16a55cd13d,6,visible_declining,refresh_review,4556,194,down
22872,content_e3ff1b093148,6,visible_declining,refresh_review,1408,183,down
11489,content_5feee3994adb,6,visible_declining,refresh_review,7812,194,down
21268,content_0a91db491d14,6,visible_declining,refresh_review,13299,193,down


## 3. Top-20 review

All of the top 20 rows are tied at the maximum score, since they're all visible AND declining. Reviewing 10 representative rows below, including a mix of different impression levels within that tied group.

In [6]:
top10 = queue.head(10).copy()

for i, row in top10.iterrows():
    print(f"- {row['content_id']}: action={row['action']}, reason={row['reason_code']}")
    print(f"  impressions_90d={row['impressions_90d']}, days_since_last_update={row['days_since_last_update']}")
    print(f"  Why flagged: visible (500+ impressions) AND currently declining.")
    print(f"  What would make this wrong: if the decline is actually seasonal or caused by a related page absorbing traffic, rather than a real quality problem with this specific page.")
    print()

- content_b16bd7307b39: action=refresh_review, reason=visible_declining
  impressions_90d=4590, days_since_last_update=194
  Why flagged: visible (500+ impressions) AND currently declining.
  What would make this wrong: if the decline is actually seasonal or caused by a related page absorbing traffic, rather than a real quality problem with this specific page.

- content_1bfaa38ff26c: action=refresh_review, reason=visible_declining
  impressions_90d=25715, days_since_last_update=194
  Why flagged: visible (500+ impressions) AND currently declining.
  What would make this wrong: if the decline is actually seasonal or caused by a related page absorbing traffic, rather than a real quality problem with this specific page.

- content_074ba6ead17b: action=refresh_review, reason=visible_declining
  impressions_90d=533, days_since_last_update=183
  Why flagged: visible (500+ impressions) AND currently declining.
  What would make this wrong: if the decline is actually seasonal or caused by a r

## 4. Weak picks + leakage check

**Weak pick pattern:** all top-ranked pages tie at the same max score (visible + declining = 6), meaning my rule can't distinguish urgency *within* that top group — a page with 500 impressions and a page with 60,000 impressions score identically. That's a real limitation: the rule flags the right broad group but can't finely rank within it. A smarter model should be able to use the actual impression count (not just a yes/no threshold) to break these ties.

**Leakage check:** I only used impressions_90d, days_since_last_update, and trend_direction — all real, observed signals available before any decision would be made. I did not use any product decision flags (health_score, priority_score, action_type) since those aren't in the dataset. I also didn't use any future-window data — trend_direction reflects the current window only, not something computed from outcomes after the fact. No leakage identified.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.